# GPT-from-Scratch: Turkish News Generator

**Demo Notebook**  
A GPT model built from scratch, trained on the Havadis Turkish news dataset.

In [ ]:
import math
import os
import sys

import torch

sys.path.insert(0, '..')

from config.config import Config
from src.bigram import BigramModel
from src.data import DataProcessor
from src.model import GPTLanguageModel

print(f"Device: {Config.device}")
print(f"Mode: {Config.mode}")

In [ ]:
# Load data and model
data_processor = DataProcessor(Config.input_path, val_split=Config.havadis_val_split)
print(f"Vocab size: {data_processor.vocab_size}")
print(f"Chars: {''.join(data_processor.chars[:50])}...")

In [ ]:
# Load best GPT checkpoint
import glob

checkpoints = sorted(glob.glob(f"../outputs/model_gpt_{Config.mode}_*.pth"))
if checkpoints:
    best_ckpt = checkpoints[-1]
    print(f"Loading: {best_ckpt}")
    model = GPTLanguageModel(data_processor.vocab_size).to(Config.device)
    ckpt = torch.load(best_ckpt, map_location=Config.device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    print(f"Val loss: {ckpt['val_loss']:.4f} (ppl {math.exp(ckpt['val_loss']):.2f})")
else:
    print("No checkpoint found. Train first with: python src/main.py")

In [ ]:
# Generation function
def generate(model, prompt, max_tokens=400, temperature=0.8, top_k=50):
    context = torch.tensor([data_processor.encode(prompt)], dtype=torch.long, device=Config.device)
    for _ in range(max_tokens):
        idx_cond = context[:, -Config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        if top_k:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, -1:]] = float('-inf')
        probs = torch.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        context = torch.cat((context, idx_next), dim=1)
    return data_processor.decode(context[0].tolist())

In [ ]:
# Generate sample news
prompts = [
    "Başlık: Yapay Zeka ve Gelecek\nİçerik:",
    "Başlık: Türkiye Ekonomisi\nİçerik:",
    "Başlık: Yeni Bir Keşif\nİçerik:",
]

for prompt in prompts:
    print(f"\n{'='*60}")
    print(f"PROMPT: {prompt}")
    print(f"{'='*60}")
    gen = generate(model, prompt, max_tokens=300, temperature=0.7)
    # Show first 600 chars after prompt
    print(gen[len(prompt):len(prompt)+600])

In [ ]:
# ATTENTION VISUALIZATION
# This shows what the model focuses on when generating each token

import matplotlib.pyplot as plt
import numpy as np


def get_attention_patterns(model, text):
    """Extract attention patterns from all heads in all layers"""
    idx = torch.tensor([data_processor.encode(text)], dtype=torch.long, device=Config.device)
    _, T = idx.shape

    # Forward pass hook
    attentions = []

    def hook_fn(name):
        def hook(module, input, output):
            if hasattr(module, 'heads'):
                head_attentions = []
                for head in module.heads:
                    with torch.no_grad():
                        x = input[0]
                        B2, T2, C2 = x.shape
                        k = head.key(x)
                        q = head.query(x)
                        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5
                        wei = wei.masked_fill(head.tril[:T2, :T2] == 0, float('-inf'))
                        wei = torch.softmax(wei, dim=-1)
                        head_attentions.append(wei[0].cpu().numpy())
                attentions.append(np.array(head_attentions))
        return hook

    hooks = []
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Module) and hasattr(module, 'heads'):
            hooks.append(module.register_forward_hook(hook_fn(name)))

    with torch.no_grad():
        model(idx)

    for h in hooks:
        h.remove()

    return np.array(attentions)

sample_text = "Başlık: Yapay Zeka"
attn = get_attention_patterns(model, sample_text)
print(f"Attention shape: {attn.shape}")  # (n_layers, n_heads, T, T)

In [ ]:
# Plot attention heatmaps for each layer
n_layers = attn.shape[0]
n_heads = attn.shape[1]
T = attn.shape[2]

fig, axes = plt.subplots(n_layers, n_heads, figsize=(n_heads*3, n_layers*3))
if n_layers == 1:
    axes = axes.reshape(1, -1)
if n_heads == 1:
    axes = axes.reshape(-1, 1)

for layer in range(n_layers):
    for head in range(n_heads):
        ax = axes[layer, head]
        im = ax.imshow(attn[layer, head], cmap='viridis', aspect='auto')
        ax.set_title(f'Layer {layer+1}, Head {head+1}', fontsize=9)
        ax.axis('off')

plt.suptitle(f'Attention Patterns for: "{sample_text}"', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/attention_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to outputs/attention_patterns.png")

In [ ]:
# COMPARISON TABLE
print("="*50)
print("MODEL COMPARISON")
print("="*50)

random_loss = math.log(data_processor.vocab_size)

# Load bigram results
if os.path.exists('../outputs/bigram_baseline.pth'):
    bigram_ckpt = torch.load('../outputs/bigram_baseline.pth', map_location='cpu')
    bigram_val = bigram_ckpt['val_loss']
else:
    bigram_val = 0

print(f"{'Model':<25} {'Loss':<10} {'PPL':<10} {'Params':<10}")
print("- "*25)
print(f"{'Random (uniform)':<25} {random_loss:<10.4f} {math.exp(random_loss):<10.2f} {'0':<10}")
if bigram_val:
    bigram_params = sum(p.numel() for p in BigramModel(data_processor.vocab_size).parameters())
    print(f"{'Bigram baseline':<25} {bigram_val:<10.4f} {math.exp(bigram_val):<10.2f} {bigram_params/1e6:<10.2f}M")
gpt_params = sum(p.numel() for p in model.parameters())
print(f"{'GPT-'+Config.mode.capitalize():<25} {ckpt['val_loss']:<10.4f} {math.exp(ckpt['val_loss']):<10.2f} {gpt_params/1e6:<10.2f}M")
print("- "*25)
print(f"\nGPT vs Bigram loss improvement: {(bigram_val - ckpt['val_loss'])/bigram_val*100:.1f}%" if bigram_val else "")

In [ ]:
# Generate a full news article from scratch
print("="*60)
print("FULL GENERATION - News article from scratch")
print("="*60)

gen = generate(model, "", max_tokens=800, temperature=0.8, top_k=40)
# Find the first complete news article
if "HABER SONU" in gen:
    idx = gen.index("HABER SONU")
    print(gen[:idx+10])
else:
    print(gen[:1000])

In [ ]:
print("Demo complete.")
print(f"Model: GPT-{Config.mode.capitalize()} | {gpt_params/1e6:.1f}M params | Havadis dataset")